# RetinaAI Triage: model notebook

Research/educational prototype, not for clinical use.

**Kaggle settings (right panel):** Accelerator **GPU T4**, Internet **On**, Input **APTOS 2019 Blindness Detection**
(accept the rules once on the competition's Data tab).

| Section | What it does | Time |
|---|---|---|
| 1 | Setup | 1 min |
| 2 | Train baseline (skipped if it already exists) | ~15 min |
| 3 | Results | – |
| 4 | Export checks for the backend | 1 min |
| 5 | Live test: true vs predicted | 1 min |
| 6 | Triage rules analysis (T / U) | seconds |
| 7 | Robustness to low-quality images: before, fine-tune, after | ~20 min |
| 8 | Download results | 1 min |

## 1 · Setup

In [ ]:
REPO = "https://github.com/arianaak/retinaai-triage.git"
DATA = "/kaggle/input/competitions/aptos2019-blindness-detection"
WORK = "/kaggle/working"
CODE = f"{WORK}/retinaai-triage"
OUT  = f"{WORK}/outputs"

import os, sys
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
%cd {CODE}
sys.path.insert(0, f"{CODE}/src")
os.environ["PYTHONPATH"] = f"{CODE}/src"          # so `python -m retinaai...` works below

assert os.path.exists(f"{DATA}/train.csv"), "APTOS data not found: add it under Input"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git log --oneline -1

## 2 · Train the baseline
Skipped automatically if `outputs/baseline/model_inference.pt` already exists in this session.
Set `RETRAIN = True` to force it. For a quick pipeline test, use `--max_samples 600 --epochs 2 --img_size 256`.

In [ ]:
RETRAIN = False
if RETRAIN or not os.path.exists(f"{OUT}/baseline/model_inference.pt"):
    !python -m retinaai.train --data_dir {DATA} --epochs 10 --img_size 384 --batch_size 32 \
        --smartphone_aug_p 0.3 --output_dir {OUT}/baseline
else:
    print("Baseline already trained, skipping.")

## 3 · Results

In [ ]:
import json, numpy as np, pandas as pd
hist = pd.DataFrame(json.load(open(f"{OUT}/baseline/history.json")))
display(hist[["epoch", "train_loss", "qwk", "accuracy", "referable_sensitivity",
              "referable_specificity", "referable_auc"]].round(3))
best = hist.loc[hist.qwk.idxmax()]
cm = pd.DataFrame(best.confusion_matrix, index=[f"true {i}" for i in range(5)],
                  columns=[f"pred {i}" for i in range(5)])
print(f"Best epoch {int(best.epoch)}: QWK {best.qwk:.3f}")
display(cm)

## 4 · Export checks (for the backend handoff)

In [ ]:
import hashlib, torch, timm, cv2
sys.path.insert(0, f"{CODE}/export"); import inference as inf

MODEL = f"{OUT}/baseline/model_inference.pt"
print("torch", torch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__, "| numpy", np.__version__)
print("sha256", hashlib.sha256(open(MODEL, "rb").read()).hexdigest())
!ls -lh {MODEL}

# inference.py (CPU) must reproduce the predictions saved during training
m   = inf.load_model(MODEL)
val = pd.read_csv(f"{OUT}/baseline/val_split.csv", dtype={"id": str})
ref = np.load(f"{OUT}/baseline/val_probs.npy")
diffs, agree = [], 0
for i in range(50):
    rgb = cv2.cvtColor(cv2.imread(f"{DATA}/train_images/{val.id[i]}.png"), cv2.COLOR_BGR2RGB)
    out = inf.predict(m, rgb)
    diffs.append(np.abs(np.array(out["probabilities"]) - ref[i]).max()); agree += out["grade"] == ref[i].argmax()
print(f"max prob difference {max(diffs):.4f} (expect < ~0.02) | grade agreement {agree}/50")

## 5 · Live test: true grade vs model, on validation images (never trained on)
✓ exact grade · ~ wrong grade but same refer/don't-refer decision · ✗ wrong referral decision

In [ ]:
from retinaai.predict import Predictor
from retinaai import GRADE_NAMES

N = 30
p = Predictor(MODEL)
sample = val.sample(N, random_state=0)
print(f"{'#':>3}  {'TRUE':<18} {'PREDICTED':<18} {'P(ref)':>6}  {'TRIAGE':<13} {'QUALITY':<8}")
n_exact = n_ref = 0
for i, r in enumerate(sample.itertuples(), 1):
    res = p.predict(f"{DATA}/train_images/{r.id}.png", return_heatmap=False)
    true, pred, pref = int(r.label), res["dr"]["grade"], res["triage"]["p_referable"]
    exact, ref_ok = pred == true, (true >= 2) == (pref >= 0.5)
    n_exact += exact; n_ref += ref_ok
    mark = "✓" if exact else ("~" if ref_ok else "✗")
    print(f"{i:>3}  {true} {GRADE_NAMES[true]:<16} {pred} {GRADE_NAMES[pred]:<16} {pref:>6.2f}  "
          f"{res['triage']['category']:<13} {res['quality']['label']:<8} {mark}   exact {n_exact/i:.0%} | referral {n_ref/i:.0%}")

## 6 · Triage rules analysis
Simulates the backend's rules (review: confidence < 0.6 or top-two gap < 0.15) plus the safety rules
T (refer predicted grade 0–1 if p2+p3+p4 ≥ T) and U (urgent for predicted grade 2 if p3+p4 ≥ U).
Chosen values: **T = 0.15, U = 0.25** (see `docs/BACKEND_HANDOFF.md` §C10).

In [ ]:
!python scripts/triage_analysis.py --run {OUT}/baseline --T 0.15 --U 0.25

## 7 · Robustness to low-quality / smartphone-like images

The same validation images at 3 levels of simulated degradation (0 = clean, 0.5 = moderate, 1.0 = strong):
narrow field of view, blur, uneven light, glare, colour cast, noise, JPEG.
Degradation is identical for every model, so the before/after comparison is fair.

**7a · Before:** the current baseline model

In [ ]:
!python -m retinaai.evaluate --checkpoint {OUT}/baseline/model_inference.pt \
    --split_csv {OUT}/baseline/val_split.csv --data_dir {DATA} \
    --out {OUT}/robustness_baseline.json --examples_png {OUT}/degradation_examples.png

from PIL import Image
print("Rows: 3 example images · Columns: clean | moderate | strong degradation")
display(Image.open(f"{OUT}/degradation_examples.png").resize((3 * 256, 3 * 256)))

**7b · Fine-tune for robustness** (starts from the baseline weights, more degraded images during training, ~6 min)

In [ ]:
if RETRAIN or not os.path.exists(f"{OUT}/robust/model_inference.pt"):
    !python -m retinaai.train --data_dir {DATA} --init_from {OUT}/baseline/model_inference.pt \
        --epochs 4 --lr 1e-4 --warmup_epochs 0.2 --img_size 384 --batch_size 32 \
        --smartphone_aug_p 0.6 --output_dir {OUT}/robust
else:
    print("Robust model already trained, skipping.")

**7c · After:** the fine-tuned model on exactly the same degraded images

In [ ]:
!python -m retinaai.evaluate --checkpoint {OUT}/robust/model_inference.pt \
    --split_csv {OUT}/baseline/val_split.csv --data_dir {DATA} --out {OUT}/robustness_robust.json

**7d · Before vs after**

In [ ]:
import matplotlib.pyplot as plt
before = pd.DataFrame(json.load(open(f"{OUT}/robustness_baseline.json"))["results"]).T
after  = pd.DataFrame(json.load(open(f"{OUT}/robustness_robust.json"))["results"]).T
cols = ["qwk", "referable_auc", "sens@0.15", "spec@0.15", "quality_reject_rate"]
comp = pd.concat({"baseline": before[cols], "robust": after[cols]}, axis=1).astype(float).round(3)
comp.index.name = "degradation"
display(comp)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, metric in zip(axes, ["qwk", "referable_auc", "sens@0.15"]):
    x = np.arange(len(before))
    ax.bar(x - 0.2, before[metric].astype(float), 0.4, label="baseline")
    ax.bar(x + 0.2, after[metric].astype(float), 0.4, label="robust")
    ax.set_xticks(x, ["clean", "moderate", "strong"]); ax.set_title(metric); ax.set_ylim(0, 1)
axes[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/robustness_comparison.png", dpi=150); plt.show()

## 8 · Download results
Bundles everything small (model files, predictions, metrics, figures). Leaves out `last.pt` and the image cache.

In [ ]:
import shutil
from IPython.display import FileLink
bundle = f"{WORK}/results_bundle"
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(OUT, bundle, ignore=shutil.ignore_patterns("last.pt"))
shutil.make_archive(f"{WORK}/results_bundle", "zip", bundle)
%cd {WORK}
display(FileLink("results_bundle.zip"))
%cd {CODE}